# Z01 Stage 2 robust walking on Kaggle

Set **Accelerator** to **GPU T4 x2** and **Internet** to **On**.
Training uses both T4s in one PPO run. Keep this
notebook private. In **Add-ons → Secrets**, add and enable
`ZLAB_GITHUB_TOKEN`, a GitHub token with read access to the Zlab-cloud repository.

Run the cells in order. Use **Save Version → Save & Run All** to retain the
checkpoints and TensorBoard logs as notebook outputs.

In [ ]:
from pathlib import Path
import base64
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

from kaggle_secrets import UserSecretsClient

repo = Path("/tmp/Zlab-cloud")
gym = repo / "Zlab_Gym"
log_root = Path("/kaggle/working/zlab-training")

def fetch_repo():
    token = UserSecretsClient().get_secret("ZLAB_GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_env = {
        **os.environ,
        "GIT_TERMINAL_PROMPT": "0",
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {auth}",
    }
    if (repo / ".git").is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], env=git_env, check=True)
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main",
             "https://github.com/hafizadha/Zlab-cloud.git", str(repo)],
            env=git_env, check=True,
        )

fetch_repo()
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(gym),
     "onnx", "tensorboard", "torchrunx==0.4.0"],
    check=True,
)


The defaults use 1,025 environments per T4 (2,050 total) for 1,500 additional
PPO updates, with saves every 500 updates and at the end. On a fresh session the launcher uses the bundled
`model_7300.pt`. In the same session it resumes the latest valid Stage 2 save.

To continue in a new session, add the previous notebook's output as input
and set `RESUME_CHECKPOINT` to the saved `model_*.pt` path under
`/kaggle/input`.

In [ ]:
UPDATES = 1500
ENVS_PER_GPU = 1025
SAVE_INTERVAL = 500
RESUME_CHECKPOINT = ""  # Optional absolute path to a Stage 2 checkpoint.

subprocess.run(["nvidia-smi", "-L"], check=True)
subprocess.run(
    [sys.executable, "-c", """
import torch

if torch.cuda.device_count() != 2 or not torch.distributed.is_nccl_available():
    raise SystemExit("Select GPU T4 x2 and use a CUDA PyTorch build with NCCL.")
for rank in range(2):
    device = f"cuda:{rank}"
    torch.ones(1, device=device).add_(1)
    torch.cuda.synchronize(device)
    print(device, torch.cuda.get_device_name(device))
"""],
    check=True,
)
for tool in ("flock", "unzip"):
    if shutil.which(tool) is None:
        raise RuntimeError(f"{tool} is required. Install unzip and util-linux in this session.")

def train_stage2(updates, envs_per_gpu, output_dir, checkpoint=""):
    python_paths = [str(gym / "src"), str(gym)]
    if os.environ.get("PYTHONPATH"):
        python_paths.append(os.environ["PYTHONPATH"])
    env = {
        **os.environ,
        "ZLAB_PYTHON": sys.executable,
        "ZLAB_USE_ALL_GPUS": "1",
        "ZLAB_SAVE_INTERVAL": str(SAVE_INTERVAL),
        "ZLAB_LOG_ROOT": str(output_dir),
        "WARP_CACHE_PATH": "/tmp/zlab-warp-cache",
        "OMP_NUM_THREADS": "2",
        "PYTHONPATH": os.pathsep.join(python_paths),
    }
    command = ["bash", "./train_walking.sh", "--robust", "2", str(updates), str(envs_per_gpu)]
    if checkpoint:
        command.append(str(Path(checkpoint).resolve(strict=True)))
    with subprocess.Popen(
        command, cwd=gym, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


Run one PPO update at `ENVS_PER_GPU` on both GPUs to check distributed training,
GPU memory, and checkpoint output. The first run can take several minutes
while MuJoCo Warp compiles GPU kernels.

In [ ]:
with tempfile.TemporaryDirectory(prefix="zlab-kaggle-smoke-") as smoke_dir:
    train_stage2(1, ENVS_PER_GPU, smoke_dir, RESUME_CHECKPOINT)
    saves = list(Path(smoke_dir).glob("rsl_rl/z01_robust_walking_2/*/model_*.pt"))
    if len(saves) != 1:
        raise RuntimeError(f"Expected one smoke-test checkpoint, found {len(saves)}.")
    with zipfile.ZipFile(saves[0]) as archive:
        if archive.testzip() is not None:
            raise RuntimeError("The smoke-test checkpoint failed its integrity check.")
print("Both GPU workers completed a PPO update and the trainer saved a checkpoint.")


Start the full training run. Open the notebook output to find checkpoints
under `zlab-training/rsl_rl/z01_robust_walking_2/`.

In [ ]:
train_stage2(UPDATES, ENVS_PER_GPU, log_root, RESUME_CHECKPOINT)
